# Step 1: clean and audit January 2024 FHV trips

This notebook preserves raw data and writes every input record to exactly one of
**accepted** or **quarantined**. Demand, simulation, fare-analysis and monetary
eligibility are separate flags: their counts may overlap and must not be added.
Repeated composite keys are flagged, never dropped or called confirmed duplicates.

Only trip cleaning is implemented. Zone-hour/zone-pair aggregation, weather and
holiday integration, forecasts, simulation and profit calculation come later.
The pre-submission report's full-year, exact-duplicate and join claims remain unverified.

Run from the repository root or `notebooks/`, using Python 3.11, Java 17/21 and
`requirements.txt`. Default: validate a bounded sample only. Set
`ADS_RUN_FULL_MONTH=1` in the kernel environment to additionally run January.
Every run uses new output folders; nothing is overwritten.

Last successful full January run: **2026-11-14, 14:25 UTC**, fresh kernel, all cells completed. Runtime versions and audit paths are printed below.

## 1. Configuration and compatibility

Paths are relative to the repo, not a teammate's home directory. UTC is used only
as a neutral Spark session setting when subtracting the source's timezone-free
timestamps; this step does not convert request times to UTC or local standard time.
Weather-time alignment is a later step. Driver memory and parallelism stay bounded.

In [1]:
# Step 1 of the ETL: configure reproducible trip cleaning without changing raw files.
import csv
import hashlib
import importlib.metadata as metadata
import json
import os
import platform
import re
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    raise RuntimeError("Run the notebook from the repo root or its notebooks folder")
sys.path.insert(0, str(PROJECT_ROOT))

TRIP_PATH = PROJECT_ROOT / "data/raw/trips/fhvhv_tripdata_2024-01.parquet"
LOOKUP_PATH = PROJECT_ROOT / "data/raw/zones/taxi_zone_lookup.csv"
ARCHIVE_PATH = PROJECT_ROOT / "data/raw/zones/taxi_zones.zip"
EXPECTED_MONTH_ROWS = 19_663_930
REAL_SAMPLE_ROWS = 10_000
DRIVER_MEMORY = "3g"
SHUFFLE_PARTITIONS = 48
RUN_FULL_MONTH = os.environ.get("ADS_RUN_FULL_MONTH", "0") == "1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

java_result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
java_version = (java_result.stderr or java_result.stdout).splitlines()[0]
java_major = int(re.search(r'version "(\d+)', java_version).group(1))
if sys.version_info < (3, 10) or java_major not in (17, 21):
    raise RuntimeError("This environment needs Python >=3.10 and Java 17 or 21")
VERSIONS = {"python": platform.python_version(), "java": java_version}
for package in ["pyspark", "py4j", "ruff", "pytest", "pyarrow", "nbformat", "nbclient", "ipykernel"]:
    VERSIONS[package] = metadata.version(package)
if VERSIONS["pyspark"] != "4.0.1":
    raise RuntimeError("Use the tested Spark version in requirements.txt")
print(json.dumps(VERSIONS, indent=2))
print("Full-month run enabled:", RUN_FULL_MONTH)

{
  "python": "3.11.9",
  "java": "openjdk version \"21.0.11\" 2026-04-21",
  "pyspark": "4.0.1",
  "py4j": "0.10.9.9",
  "ruff": "0.14.4",
  "pytest": "9.1.1",
  "pyarrow": "24.0.0",
  "nbformat": "5.10.4",
  "nbclient": "0.10.4",
  "ipykernel": "7.2.0"
}
Full-month run enabled: True


## 2. Gate the run on synthetic tests and Ruff

The tests use invented rows, including exact filter boundaries, zero duration,
nulls, non-finite values, repeated keys, and recognised/unrecognised destinations.
Ruff checks Python and notebook code for mistakes such as undefined names. Either
failure stops this notebook before sample or full-month outputs are written.

In [2]:
for arguments in [["-m", "pytest", "tests", "-q"], ["-m", "ruff", "check", "."]]:
    check = subprocess.run(
        [sys.executable, *arguments], cwd=PROJECT_ROOT,
        capture_output=True, text=True,
    )
    print(check.stdout)
    if check.returncode:
        print(check.stderr)
        raise RuntimeError(f"Validation failed: {' '.join(arguments)}")

........................                                                 [100%]
24 passed in 22.44s

All checks passed!



## 3. Check inputs and start local Spark

Parquet footer metadata gives the expected January count without collecting the
month. Hashes are checked against our earlier inspection, not a publisher-supplied
checksum. The lookup has unique IDs 1–265; verified boundary coverage is 1–263,
with recognised unmapped IDs 264 and 265. No trip-to-zone join is performed here.

In [3]:
import pyarrow.parquet as parquet
from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src.trip_cleaning import (
    COLLISION_KEY,
    COLUMN_NAMES,
    QUALITY_FAILURE_FLAGS,
    clean_trips,
)


def hash_file(path):
    """Return a file's SHA256 using bounded reads, to verify raw data is unchanged."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


INPUT_HASHES = {str(path.relative_to(PROJECT_ROOT)): hash_file(path)
                for path in [TRIP_PATH, LOOKUP_PATH, ARCHIVE_PATH]}
inspection_path = PROJECT_ROOT / "data/inspection/january_2024_initial_inspection.json"
inspection = json.loads(inspection_path.read_text())
for item in inspection["files"]:
    if INPUT_HASHES[item["path"]] != item["sha256"]:
        raise RuntimeError(f"Raw input changed since inspection: {item['path']}")
parquet_rows = parquet.ParquetFile(TRIP_PATH).metadata.num_rows
assert parquet_rows == EXPECTED_MONTH_ROWS
with LOOKUP_PATH.open(newline="") as source:
    known_zone_ids = [int(row["LocationID"]) for row in csv.DictReader(source)]
assert len(known_zone_ids) == len(set(known_zone_ids)) == 265
assert set(known_zone_ids) == set(range(1, 266))

spark = (
    SparkSession.builder.master("local[2]").appName("ADS-January-trip-cleaning")
    .config("spark.driver.memory", DRIVER_MEMORY)
    .config("spark.driver.maxResultSize", "128m")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config("spark.sql.ansi.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
raw_trips = spark.read.parquet(str(TRIP_PATH))
raw_trips.printSchema()
print("Input rows from Parquet footer:", parquet_rows)
print("No zone join: categories will use the verified lookup ID set.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/14 19:51:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

## 4. Audit definitions

Acceptance depends on present request/pickup times, pickup ID, duration and miles;
finite miles; duration 60–10,800 seconds; speed 1–80 mph; and request before pickup.
Speed is evaluated only for positive `trip_time`. Missing optional data is retained.
Missing destination, drop-off time or service licence does not alone erase otherwise
usable pickup demand. These rows have separate simulation/key-completeness flags.

Both mapped endpoints and usable drop-off timing are needed for simulation.
Monetary eligibility uses base fare and tolls; fare-analysis eligibility uses base
fare alone. Missing/negative/non-finite monetary values remain unchanged. Tips are
never revenue. A mapped pickup here means one of the 263 boundary zones (including
Newark Airport); launch-area selection is not part of cleaning.

The exact collision key is service licence + request/pickup/drop-off timestamps +
pickup/drop-off IDs. Only complete keys are compared, across the whole input scope
before partitioning. Sample collisions describe the sample, not January.

Filter counts form a sequential waterfall. Individual flags overlap; eligibility
counts are independent. Timestamp differences are signed timestamp elapsed seconds
minus `trip_time`, with any nonzero difference flagged but not used as a filter.

In [4]:
def count_trip_flags(trips, raw_column_names):
    """Return independent flag counts and null counts, without collecting trip rows."""
    boolean_names = [field.name for field in trips.schema if field.dataType.simpleString() == "boolean"]
    expressions = [F.count(F.lit(1)).alias("input_rows")]
    for name in boolean_names:
        expressions.append(F.count(F.when(F.col(name), 1)).alias(name))
        expressions.append(F.count(F.when(F.col(name).isNull(), 1)).alias(f"null_flag_{name}"))
    for name in raw_column_names:
        expressions.append(F.count(F.when(F.col(name).isNull(), 1)).alias(f"missing_{name}"))
    return trips.agg(*expressions).first().asDict()


def count_filter_stages(trips):
    """Return a sequential filter waterfall; no physical trip rows are removed here."""
    passes_so_far = F.lit(True)
    expressions = []
    for name in QUALITY_FAILURE_FLAGS:
        passes_so_far = passes_so_far & ~F.col(name)
        expressions.append(F.count(F.when(passes_so_far, 1)).alias(name))
    return trips.agg(*expressions).first().asDict()


def describe_timestamp_differences(trips):
    """Return signed difference statistics and disjoint absolute-difference bands."""
    difference = F.col("duration_difference_seconds")
    absolute = F.abs(difference)
    bands = {
        "unavailable": difference.isNull(), "exact_match": difference == 0,
        "above_0_through_1_second": (absolute > 0) & (absolute <= 1),
        "above_1_through_60_seconds": (absolute > 1) & (absolute <= 60),
        "above_60_through_300_seconds": (absolute > 60) & (absolute <= 300),
        "above_300_seconds": absolute > 300,
    }
    expressions = [
        F.min(difference).alias("min_signed_seconds"),
        F.max(difference).alias("max_signed_seconds"),
        F.avg(difference).alias("mean_signed_seconds"),
        F.percentile_approx(difference, [0.5, 0.9, 0.99], 10_000).alias("signed_p50_p90_p99"),
        F.percentile_approx(absolute, [0.5, 0.9, 0.99], 10_000).alias("absolute_p50_p90_p99"),
        F.count(F.when(difference < 0, 1)).alias("negative_differences"),
        F.count(F.when(difference > 0, 1)).alias("positive_differences"),
    ]
    for name, condition in bands.items():
        expressions.append(F.count(F.when(condition, 1)).alias(name))
    summary = trips.agg(*expressions).first().asDict()
    summary["band_total"] = sum(summary[name] for name in bands)
    summary["quantile_method"] = "Spark percentile_approx, accuracy 10000"
    return summary


def count_key_groups(trips):
    """Count groups exactly from each row's group size; preserve incomplete keys separately."""
    sizes = trips.filter(F.col("key_record_count").isNotNull()).groupBy("key_record_count").count()
    distribution = []
    for row in sizes.orderBy("key_record_count").collect():
        assert row["count"] % row.key_record_count == 0
        distribution.append({"records_per_key": row.key_record_count,
                             "records": row["count"],
                             "key_groups": row["count"] // row.key_record_count})
    return distribution


def count_zone_categories(trips):
    """Return pickup and destination categories separately, including unmapped/missing IDs."""
    categories = {}
    for endpoint in ["pickup", "dropoff"]:
        categories[endpoint] = [row.asDict() for row in
                                trips.groupBy(f"{endpoint}_zone_status").count()
                                .orderBy(f"{endpoint}_zone_status").collect()]
    return categories

## 5. Run one scope, reconcile and save without overwriting

Spark stores annotated trips on disk during audits instead of collecting the month
in Python RAM. The single output dataset is partitioned by `cleaning_status`:
accepted and quarantined are disjoint and exhaustive. Every repeated-key row is
written too. Read-back counts must match input counts and independent eligibility
counts. Audit JSON records versions, scope and hashes. Failed runs keep their output
folders for investigation; they are not marked complete.

In [5]:
def run_cleaning_scope(raw_scope, expected_rows, scope_name):
    """Clean one scope, audit it, and save a lossless partition in a fresh folder."""
    run_id = f"{scope_name}-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}-{uuid.uuid4().hex[:8]}"
    output_path = PROJECT_ROOT / "data/processed/trip_cleaning" / run_id / "trips"
    audit_path = PROJECT_ROOT / "data/audits/trip_cleaning" / f"{run_id}.json"
    raw_names = [COLUMN_NAMES.get(name, name) for name in raw_scope.columns]
    trips = clean_trips(raw_scope, known_zone_ids).persist(StorageLevel.DISK_ONLY)
    try:
        counts = count_trip_flags(trips, raw_names)
        assert counts["input_rows"] == expected_rows, "Flagging changed the row count"
        assert all(value == 0 for name, value in counts.items() if name.startswith("null_flag_"))
        waterfall = count_filter_stages(trips)
        assert waterfall[QUALITY_FAILURE_FLAGS[-1]] == counts["is_accepted"]
        timestamp_summary = describe_timestamp_differences(trips)
        assert timestamp_summary["band_total"] == expected_rows
        keys = count_key_groups(trips)
        assert sum(item["records"] for item in keys) + counts["has_incomplete_key"] == expected_rows
        assert sum(item["records"] for item in keys if item["records_per_key"] > 1) == counts["has_repeated_key"]
        zone_categories = count_zone_categories(trips)
        for endpoint_counts in zone_categories.values():
            assert sum(item["count"] for item in endpoint_counts) == expected_rows
        combinations = [row.asDict() for row in trips.groupBy(
            "cleaning_status", "is_demand_eligible", "is_simulation_eligible",
            "is_fare_analysis_eligible", "is_monetary_eligible",
        ).count().collect()]
        trips.write.mode("errorifexists").partitionBy("cleaning_status").parquet(str(output_path))
    finally:
        trips.unpersist()

    saved = spark.read.parquet(str(output_path))
    partition_counts = {row.cleaning_status: row["count"] for row in saved.groupBy("cleaning_status").count().collect()}
    accepted = partition_counts.get("accepted", 0)
    quarantined = partition_counts.get("quarantined", 0)
    assert set(partition_counts) <= {"accepted", "quarantined"}
    assert accepted + quarantined == expected_rows
    assert accepted == counts["is_accepted"]
    verification_flags = ["has_repeated_key", "is_demand_eligible", "is_simulation_eligible",
                          "is_fare_analysis_eligible", "is_monetary_eligible"]
    saved_counts = saved.agg(*[F.count(F.when(F.col(name), 1)).alias(name)
                               for name in verification_flags]).first().asDict()
    for name in verification_flags:
        assert saved_counts[name] == counts[name], f"Read-back count changed: {name}"
    audit = {
        "scope": scope_name, "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "versions": VERSIONS, "input_sha256": INPUT_HASHES,
        "expected_rows": expected_rows, "counts": counts,
        "sequential_rows_remaining": waterfall, "zone_categories": zone_categories,
        "collision_key": COLLISION_KEY, "collision_group_size_distribution": keys,
        "timestamp_differences": timestamp_summary,
        "eligibility_combinations": combinations,
        "partition_counts": {"accepted": accepted, "quarantined": quarantined},
        "read_back_flag_counts": saved_counts,
        "output_path": str(output_path.relative_to(PROJECT_ROOT)),
        "raw_columns": raw_names, "status": "complete",
        "unverified_report_claims": ["full-year totals", "exact duplicates", "original collision-key rate",
                                      "trip-zone join coverage", "weather join coverage"],
    }
    audit_path.parent.mkdir(parents=True, exist_ok=True)
    with audit_path.open("x") as output:
        json.dump(audit, output, indent=2)
    print("Saved:", output_path.relative_to(PROJECT_ROOT))
    print("Audit:", audit_path.relative_to(PROJECT_ROOT))
    print("Partition:", audit["partition_counts"])
    print("Independent eligibility:", {name: counts[name] for name in verification_flags[1:]})
    print("Repeated-key records:", counts["has_repeated_key"])
    print("Filter waterfall:", waterfall)
    print("Timestamp differences:", timestamp_summary)
    return audit

## 6. Bounded real-data validation

Only 10,000 raw rows are passed to sample cleaning. All sample reconciliation
assertions and saved-output checks must pass before the full-month cell can run.
This is a bounded check of real input, not a representative statistical sample.
Synthetic tests cover edge cases that this slice might not contain.

In [6]:
sample_audit = run_cleaning_scope(raw_trips.limit(REAL_SAMPLE_ROWS), REAL_SAMPLE_ROWS, "sample")
assert sample_audit["status"] == "complete"
print("Sample validation passed. No full-month result is inferred from sample rates.")

26/11/14 19:51:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Saved: data/processed/trip_cleaning/sample-20261114T142103-f59418f7/trips
Audit: data/audits/trip_cleaning/sample-20261114T142103-f59418f7.json
Partition: {'accepted': 9921, 'quarantined': 79}
Independent eligibility: {'is_demand_eligible': 9921, 'is_simulation_eligible': 9509, 'is_fare_analysis_eligible': 9921, 'is_monetary_eligible': 9921}
Repeated-key records: 0
Filter waterfall: {'has_missing_core_field': 10000, 'has_nonfinite_trip_miles': 10000, 'has_nonpositive_trip_seconds': 10000, 'has_duration_outside_limits': 9998, 'has_speed_outside_limits': 9992, 'has_invalid_request_order': 9921}
Timestamp differences: {'min_signed_seconds': -1.0, 'max_signed_seconds': 0.0, 'mean_signed_seconds': -0.0004, 'signed_p50_p90_p99': [0.0, 0.0, 0.0], 'absolute_p50_p90_p99': [0.0, 0.0, 0.0], 'negative_differences': 4, 'positive_differences': 0, 'unavailable': 0, 'exact_match': 9996, 'above_0_through_1_second': 4, 'above_1_through_60_seconds': 0, 'above_60_through_300_seconds': 0, 'above_300_second

## 7. January run (only after tests, Ruff and sample checks pass)

The full run must reconcile all 19,663,930 input records. Its audit reproduces only
this month's results under the explicitly agreed key and cleaning rules. It does
not retroactively verify the pre-submission report's different, undocumented audit.

In [7]:
month_audit = None
if RUN_FULL_MONTH:
    assert sample_audit["status"] == "complete"
    month_audit = run_cleaning_scope(raw_trips, EXPECTED_MONTH_ROWS, "january-2024")
    assert sum(month_audit["partition_counts"].values()) == EXPECTED_MONTH_ROWS
else:
    print("Full month skipped. Set ADS_RUN_FULL_MONTH=1 to enable the approved January run.")

Saved: data/processed/trip_cleaning/january-2024-20261114T142109-e471127c/trips
Audit: data/audits/trip_cleaning/january-2024-20261114T142109-e471127c.json
Partition: {'accepted': 19468657, 'quarantined': 195273}
Independent eligibility: {'is_demand_eligible': 19467915, 'is_simulation_eligible': 18738600, 'is_fare_analysis_eligible': 19468483, 'is_monetary_eligible': 19468483}
Repeated-key records: 2
Filter waterfall: {'has_missing_core_field': 19663930, 'has_nonfinite_trip_miles': 19663930, 'has_nonpositive_trip_seconds': 19663928, 'has_duration_outside_limits': 19661499, 'has_speed_outside_limits': 19655760, 'has_invalid_request_order': 19468657}
Timestamp differences: {'min_signed_seconds': -1.0, 'max_signed_seconds': 0.0, 'mean_signed_seconds': -0.0004978658894737725, 'signed_p50_p90_p99': [0.0, 0.0, 0.0], 'absolute_p50_p90_p99': [0.0, 0.0, 0.0], 'negative_differences': 9790, 'positive_differences': 0, 'unavailable': 0, 'exact_match': 19654140, 'above_0_through_1_second': 9790, 'ab

## 8. Verify raw preservation and finish

The hashes must still match. The final message is printed only after the run
reaches this cell; the audit and notebook outputs show which scope actually ran.
Do not commit any files in `data/`. This notebook stops after cleaning.

In [8]:
for path in [TRIP_PATH, LOOKUP_PATH, ARCHIVE_PATH]:
    assert hash_file(path) == INPUT_HASHES[str(path.relative_to(PROJECT_ROOT))]
spark.stop()
print("Raw hashes unchanged. Notebook reached the end without errors.")
print("Completed scopes:", ["sample", "january-2024"] if month_audit else ["sample"])

Raw hashes unchanged. Notebook reached the end without errors.
Completed scopes: ['sample', 'january-2024']
